# Modulo 4 · Lezione 7 — Progetto: Sistema di inventario

**Corso di Intelligenza Artificiale — Livello base**

---

## Cosa costruiremo?

Un sistema di gestione dell'inventario completo che integra tutto ciò che è stato imparato nel modulo:

```
┌─────────────────────────────────────────────────────────────────┐
│                    SISTEMA DI INVENTARIO                        │
│                                                                 │
│  ┌─────────────┐    ┌─────────────┐    ┌─────────────────────┐ │
│  │ SQLite      │    │   Python    │    │      pandas         │ │
│  │ (dati)      │◄──►│ (logica)    │◄──►│    (analisi)        │ │
│  └─────────────┘    └─────────────┘    └─────────────────────┘ │
│         ↑                  ↑                    ↑               │
│    DDL/DML/JOIN      CRUD + regole         Report CSV          │
│    normalizzazione   di business           grafici testuali    │
└─────────────────────────────────────────────────────────────────┘
```

## Caratteristiche del sistema

| # | Modulo | Funzionalità |
|---|--------|---------------|
| 1 | `BaseDeDatos` | Creare lo schema, connettersi, chiudere |
| 2 | `GestorProductos` | CRUD prodotti + categorie |
| 3 | `GestorMovimientos` | Entrate e uscite di stock |
| 4 | `GestorProveedores` | Ordini ai fornitori |
| 5 | `Analitica` | Report, allerte, esportazione CSV |
| 6 | Demo completa | Flusso di lavoro reale |

---
## Blocco 1 — Database e schema

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path
from datetime import date, datetime
from typing import Optional

DB_PATH = Path.home() / "inventario.db"


class BaseDeDatos:
    """Gestisce la connessione e lo schema del database."""

    DDL = """
        PRAGMA foreign_keys = ON;

        CREATE TABLE IF NOT EXISTS categorias (
            id      INTEGER PRIMARY KEY AUTOINCREMENT,
            nombre  TEXT NOT NULL UNIQUE,
            descripcion TEXT
        );

        CREATE TABLE IF NOT EXISTS proveedores (
            id       INTEGER PRIMARY KEY AUTOINCREMENT,
            nombre   TEXT NOT NULL,
            email    TEXT,
            telefono TEXT,
            activo   INTEGER DEFAULT 1
        );

        CREATE TABLE IF NOT EXISTS productos (
            id              INTEGER PRIMARY KEY AUTOINCREMENT,
            sku             TEXT    NOT NULL UNIQUE,
            nombre          TEXT    NOT NULL,
            descripcion     TEXT,
            precio_venta    REAL    NOT NULL CHECK(precio_venta >= 0),
            precio_coste    REAL    NOT NULL CHECK(precio_coste >= 0),
            stock_actual    INTEGER NOT NULL DEFAULT 0 CHECK(stock_actual >= 0),
            stock_minimo    INTEGER NOT NULL DEFAULT 5,
            categoria_id    INTEGER REFERENCES categorias(id),
            proveedor_id    INTEGER REFERENCES proveedores(id),
            activo          INTEGER DEFAULT 1
        );

        CREATE TABLE IF NOT EXISTS movimientos (
            id          INTEGER PRIMARY KEY AUTOINCREMENT,
            producto_id INTEGER NOT NULL REFERENCES productos(id),
            tipo        TEXT NOT NULL CHECK(tipo IN ('entrada','salida','ajuste')),
            cantidad    INTEGER NOT NULL,
            motivo      TEXT,
            fecha       TEXT NOT NULL DEFAULT (date('now')),
            usuario     TEXT DEFAULT 'sistema'
        );

        CREATE TABLE IF NOT EXISTS pedidos_proveedor (
            id              INTEGER PRIMARY KEY AUTOINCREMENT,
            proveedor_id    INTEGER NOT NULL REFERENCES proveedores(id),
            fecha_pedido    TEXT NOT NULL DEFAULT (date('now')),
            fecha_esperada  TEXT,
            estado          TEXT NOT NULL DEFAULT 'pendiente'
                            CHECK(estado IN ('pendiente','recibido','cancelado')),
            notas           TEXT
        );

        CREATE TABLE IF NOT EXISTS lineas_pedido (
            pedido_id   INTEGER NOT NULL REFERENCES pedidos_proveedor(id),
            producto_id INTEGER NOT NULL REFERENCES productos(id),
            cantidad    INTEGER NOT NULL CHECK(cantidad > 0),
            precio_ud   REAL    NOT NULL,
            PRIMARY KEY (pedido_id, producto_id)
        );
    """

    def __init__(self, path: Path = DB_PATH):
        self.path = path
        self.conn = sqlite3.connect(str(path))
        self.conn.row_factory = sqlite3.Row
        self.conn.executescript(self.DDL)
        self.conn.commit()

    def q(self, sql, params=()):
        return pd.read_sql_query(sql, self.conn, params=params)

    def ejecutar(self, sql, params=()):
        with self.conn:
            cur = self.conn.execute(sql, params)
            return cur

    def cerrar(self):
        self.conn.close()


# Creare l'istanza (elimina il DB precedente se esiste)
if DB_PATH.exists():
    DB_PATH.unlink()

bd = BaseDeDatos()
print(f"Database creato in {DB_PATH}")
tablas = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", bd.conn)
print("Tabelle:", tablas['name'].tolist())

---
## Blocco 2 — Gestione dei prodotti

In [ ]:
class GestorProductos:

    def __init__(self, bd: BaseDeDatos):
        self.bd = bd

    # ── Categorie ───────────────────────────────────────────────

    def crear_categoria(self, nombre: str, descripcion: str = None) -> int:
        cur = self.bd.ejecutar(
            "INSERT INTO categorias (nombre, descripcion) VALUES (?, ?)",
            (nombre, descripcion)
        )
        return cur.lastrowid

    def listar_categorias(self) -> pd.DataFrame:
        return self.bd.q("SELECT * FROM categorias ORDER BY nombre")

    # ── Fornitori ───────────────────────────────────────────────

    def crear_proveedor(self, nombre: str, email: str = None, telefono: str = None) -> int:
        cur = self.bd.ejecutar(
            "INSERT INTO proveedores (nombre, email, telefono) VALUES (?, ?, ?)",
            (nombre, email, telefono)
        )
        return cur.lastrowid

    # ── Prodotti ────────────────────────────────────────────────

    def crear_producto(
        self, sku: str, nombre: str,
        precio_venta: float, precio_coste: float,
        categoria_id: int, proveedor_id: int,
        stock_inicial: int = 0, stock_minimo: int = 5,
        descripcion: str = None
    ) -> int:
        if precio_coste >= precio_venta:
            raise ValueError(f"Il prezzo di costo ({precio_coste}€) deve essere minore di quello di vendita ({precio_venta}€)")

        cur = self.bd.ejecutar("""
            INSERT INTO productos
                (sku, nombre, descripcion, precio_venta, precio_coste,
                 stock_actual, stock_minimo, categoria_id, proveedor_id)
            VALUES (?,?,?,?,?,?,?,?,?)
        """, (sku, nombre, descripcion, precio_venta, precio_coste,
              stock_inicial, stock_minimo, categoria_id, proveedor_id))

        if stock_inicial > 0:
            self.bd.ejecutar("""
                INSERT INTO movimientos (producto_id, tipo, cantidad, motivo)
                VALUES (?, 'entrada', ?, 'Stock inicial')
            """, (cur.lastrowid, stock_inicial))

        return cur.lastrowid

    def obtener_producto(self, producto_id: int) -> Optional[dict]:
        cur = self.bd.conn.execute("""
            SELECT p.*, c.nombre AS categoria, pr.nombre AS proveedor
            FROM productos p
            LEFT JOIN categorias c  ON p.categoria_id = c.id
            LEFT JOIN proveedores pr ON p.proveedor_id = pr.id
            WHERE p.id = ?
        """, (producto_id,))
        fila = cur.fetchone()
        return dict(fila) if fila else None

    def listar_productos(self, solo_activos: bool = True) -> pd.DataFrame:
        where = "WHERE p.activo = 1" if solo_activos else ""
        return self.bd.q(f"""
            SELECT p.id, p.sku, p.nombre,
                   c.nombre AS categoria,
                   p.precio_venta, p.precio_coste,
                   ROUND((p.precio_venta - p.precio_coste) / p.precio_venta * 100, 1) AS margen_pct,
                   p.stock_actual, p.stock_minimo,
                   CASE WHEN p.stock_actual <= p.stock_minimo THEN '⚠️' ELSE '✅' END AS alerta
            FROM productos p
            LEFT JOIN categorias c ON p.categoria_id = c.id
            {where}
            ORDER BY c.nombre, p.nombre
        """)

    def actualizar_precio(self, producto_id: int, nuevo_precio_venta: float):
        self.bd.ejecutar(
            "UPDATE productos SET precio_venta = ? WHERE id = ?",
            (nuevo_precio_venta, producto_id)
        )

    def desactivar_producto(self, producto_id: int):
        self.bd.ejecutar("UPDATE productos SET activo = 0 WHERE id = ?", (producto_id,))


# Istanziare e provare
gp = GestorProductos(bd)
print("GestorProductos pronto")

In [ ]:
# Creare dati di esempio

# Categorie
id_elec = gp.crear_categoria("Electrónica",   "Dispositivos y accesorios electrónicos")
id_info = gp.crear_categoria("Informática",   "Equipos y periféricos")
id_hog  = gp.crear_categoria("Hogar",         "Artículos para el hogar")
id_ofic = gp.crear_categoria("Oficina",       "Material de oficina")

# Fornitori
id_prov1 = gp.crear_proveedor("TechDistrib SL",    "pedidos@techdistrib.es",    "91 111 2222")
id_prov2 = gp.crear_proveedor("HogarSupply SA",    "compras@hogarsupply.com",   "93 333 4444")
id_prov3 = gp.crear_proveedor("OfficeWorld SL",    "proveedores@officeworld.es","95 555 6666")

# Prodotti
productos_data = [
    ("EL-001", "Smartphone X12",   599.99, 380.0,  id_elec, id_prov1, 50, 10),
    ("EL-002", "Auriculares BT",    89.99,  45.0,  id_elec, id_prov1, 120, 20),
    ("EL-003", "Cargador rápido",   29.99,  12.0,  id_elec, id_prov1, 200, 30),
    ("IN-001", "Portátil Pro 15",  1199.0,  780.0, id_info, id_prov1, 30, 5),
    ("IN-002", "Teclado mecánico",  85.0,   42.0,  id_info, id_prov1, 75, 15),
    ("IN-003", "Ratón inalámbrico", 45.0,   18.0,  id_info, id_prov1, 90, 20),
    ("IN-004", "Monitor 27 4K",    450.0,  280.0,  id_info, id_prov1, 25, 5),
    ("HO-001", "Lámpara escritorio",35.0,   14.0,  id_hog,  id_prov2, 60, 10),
    ("HO-002", "Silla ergonómica", 299.0,  180.0,  id_hog,  id_prov2, 15, 3),
    ("OF-001", "Bolígrafos 10u",     4.99,   1.50, id_ofic, id_prov3, 500, 50),
    ("OF-002", "Agenda 2025",       12.99,   4.0,  id_ofic, id_prov3, 150, 20),
    ("OF-003", "Post-it pack",       8.50,   2.5,  id_ofic, id_prov3, 300, 40),
]

ids_prod = []
for sku, nom, pv, pc, cat, prov, stock, smin in productos_data:
    pid = gp.crear_producto(sku, nom, pv, pc, cat, prov, stock, smin)
    ids_prod.append(pid)

print(f"Creati {len(ids_prod)} prodotti")
display(gp.listar_productos())

---
## Blocco 3 — Gestione dei movimenti di stock

In [ ]:
class GestorMovimientos:

    def __init__(self, bd: BaseDeDatos):
        self.bd = bd

    def entrada(self, producto_id: int, cantidad: int, motivo: str = "Compra", usuario: str = "sistema"):
        """Aggiunge stock al prodotto."""
        if cantidad <= 0:
            raise ValueError("La quantità deve essere positiva")

        with self.bd.conn:
            self.bd.conn.execute("""
                INSERT INTO movimientos (producto_id, tipo, cantidad, motivo, usuario)
                VALUES (?, 'entrada', ?, ?, ?)
            """, (producto_id, cantidad, motivo, usuario))
            self.bd.conn.execute(
                "UPDATE productos SET stock_actual = stock_actual + ? WHERE id = ?",
                (cantidad, producto_id)
            )

    def salida(self, producto_id: int, cantidad: int, motivo: str = "Venta", usuario: str = "sistema"):
        """Riduce lo stock del prodotto."""
        if cantidad <= 0:
            raise ValueError("La quantità deve essere positiva")

        cur = self.bd.conn.execute(
            "SELECT stock_actual, nombre FROM productos WHERE id = ?", (producto_id,)
        )
        prod = cur.fetchone()
        if not prod:
            raise ValueError(f"Prodotto {producto_id} non trovato")
        if prod['stock_actual'] < cantidad:
            raise ValueError(
                f"Stock insufficiente per '{prod['nombre']}': "
                f"richiesto={cantidad}, disponibile={prod['stock_actual']}"
            )

        with self.bd.conn:
            self.bd.conn.execute("""
                INSERT INTO movimientos (producto_id, tipo, cantidad, motivo, usuario)
                VALUES (?, 'salida', ?, ?, ?)
            """, (producto_id, cantidad, motivo, usuario))
            self.bd.conn.execute(
                "UPDATE productos SET stock_actual = stock_actual - ? WHERE id = ?",
                (cantidad, producto_id)
            )

    def ajuste(self, producto_id: int, nuevo_stock: int, motivo: str = "Inventario", usuario: str = "sistema"):
        """Imposta lo stock su un valore specifico."""
        cur = self.bd.conn.execute("SELECT stock_actual FROM productos WHERE id = ?", (producto_id,))
        stock_actual = cur.fetchone()['stock_actual']
        diferencia = nuevo_stock - stock_actual

        with self.bd.conn:
            self.bd.conn.execute("""
                INSERT INTO movimientos (producto_id, tipo, cantidad, motivo, usuario)
                VALUES (?, 'ajuste', ?, ?, ?)
            """, (producto_id, diferencia, motivo, usuario))
            self.bd.conn.execute(
                "UPDATE productos SET stock_actual = ? WHERE id = ?",
                (nuevo_stock, producto_id)
            )

    def historial(self, producto_id: int) -> pd.DataFrame:
        """Storico dei movimenti di un prodotto."""
        return self.bd.q("""
            SELECT m.fecha, m.tipo, m.cantidad, m.motivo, m.usuario
            FROM movimientos m
            WHERE m.producto_id = ?
            ORDER BY m.id DESC
        """, (producto_id,))

    def alertas_stock(self) -> pd.DataFrame:
        """Prodotti con stock al di sotto del minimo."""
        return self.bd.q("""
            SELECT p.sku, p.nombre, p.stock_actual, p.stock_minimo,
                   p.stock_minimo - p.stock_actual AS deficit,
                   c.nombre AS categoria,
                   pr.nombre AS proveedor
            FROM productos p
            LEFT JOIN categorias c  ON p.categoria_id = c.id
            LEFT JOIN proveedores pr ON p.proveedor_id = pr.id
            WHERE p.stock_actual <= p.stock_minimo
              AND p.activo = 1
            ORDER BY deficit DESC
        """)


gm = GestorMovimientos(bd)

# Simuliamo dei movimenti
pid1, pid2, pid3, pid4 = ids_prod[0], ids_prod[1], ids_prod[3], ids_prod[8]

# Vendite
gm.salida(pid1, 5,  "Venta online", "tienda_web")
gm.salida(pid1, 3,  "Venta tienda", "empleado1")
gm.salida(pid2, 12, "Venta online", "tienda_web")
gm.salida(pid3, 2,  "Venta corporativa", "comercial1")
gm.salida(pid4, 10, "Venta tienda", "empleado2")  # sedia ergonomica

# Riassortimento
gm.entrada(pid1, 20, "Pedido proveedor #101", "almacen")

# Rettifica inventario
gm.ajuste(ids_prod[5], 85, "Inventario mensual", "jefa_almacen")

print("Movimenti registrati")
print("\nStorico Smartphone:")
display(gm.historial(pid1))

print("\n⚠️ Allerte di stock minimo:")
alertas = gm.alertas_stock()
display(alertas)

---
## Blocco 4 — Gestione degli ordini ai fornitori

In [ ]:
class GestorProveedores:

    def __init__(self, bd: BaseDeDatos, gm: GestorMovimientos):
        self.bd = bd
        self.gm = gm

    def crear_pedido(
        self, proveedor_id: int,
        lineas: list[tuple[int, int, float]],  # [(producto_id, cantidad, precio_ud)]
        fecha_esperada: str = None,
        notas: str = None
    ) -> int:
        """Crea un ordine d'acquisto al fornitore."""
        with self.bd.conn:
            cur = self.bd.conn.execute("""
                INSERT INTO pedidos_proveedor (proveedor_id, fecha_esperada, notas)
                VALUES (?, ?, ?)
            """, (proveedor_id, fecha_esperada, notas))
            pedido_id = cur.lastrowid

            for prod_id, cantidad, precio_ud in lineas:
                self.bd.conn.execute("""
                    INSERT INTO lineas_pedido (pedido_id, producto_id, cantidad, precio_ud)
                    VALUES (?, ?, ?, ?)
                """, (pedido_id, prod_id, cantidad, precio_ud))

        return pedido_id

    def recibir_pedido(self, pedido_id: int, usuario: str = "almacen"):
        """Segna l'ordine come ricevuto e aggiunge lo stock."""
        cur = self.bd.conn.execute(
            "SELECT estado FROM pedidos_proveedor WHERE id = ?", (pedido_id,)
        )
        pedido = cur.fetchone()
        if not pedido:
            raise ValueError(f"Ordine {pedido_id} non trovato")
        if pedido['estado'] != 'pendiente':
            raise ValueError(f"L'ordine {pedido_id} non è nello stato 'pendiente'")

        lineas = self.bd.conn.execute(
            "SELECT producto_id, cantidad FROM lineas_pedido WHERE pedido_id = ?",
            (pedido_id,)
        ).fetchall()

        with self.bd.conn:
            self.bd.conn.execute(
                "UPDATE pedidos_proveedor SET estado = 'recibido' WHERE id = ?",
                (pedido_id,)
            )
            for linea in lineas:
                self.gm.entrada(
                    linea['producto_id'], linea['cantidad'],
                    f"Pedido proveedor #{pedido_id}", usuario
                )

    def listar_pedidos(self) -> pd.DataFrame:
        return self.bd.q("""
            SELECT pp.id, pr.nombre AS proveedor,
                   pp.fecha_pedido, pp.fecha_esperada, pp.estado,
                   COUNT(lp.producto_id)                   AS lineas,
                   ROUND(SUM(lp.cantidad * lp.precio_ud), 2) AS total
            FROM pedidos_proveedor pp
            JOIN proveedores pr ON pp.proveedor_id = pr.id
            JOIN lineas_pedido lp ON pp.id = lp.pedido_id
            GROUP BY pp.id, pr.nombre, pp.fecha_pedido, pp.fecha_esperada, pp.estado
            ORDER BY pp.id DESC
        """)

    def sugerir_pedidos(self) -> pd.DataFrame:
        """Suggerisce cosa ordinare in base alle allerte di stock."""
        return self.bd.q("""
            SELECT pr.nombre AS proveedor,
                   p.sku, p.nombre, p.stock_actual, p.stock_minimo,
                   (p.stock_minimo * 3 - p.stock_actual) AS cantidad_sugerida,
                   ROUND((p.stock_minimo * 3 - p.stock_actual) * p.precio_coste, 2) AS coste_estimado
            FROM productos p
            JOIN proveedores pr ON p.proveedor_id = pr.id
            WHERE p.stock_actual <= p.stock_minimo AND p.activo = 1
            ORDER BY coste_estimado DESC
        """)


gprov = GestorProveedores(bd, gm)

# Creare e ricevere un ordine
id_pedido = gprov.crear_pedido(
    id_prov1,
    lineas=[
        (ids_prod[8], 20, 165.0),  # sedie ergonomiche
        (ids_prod[3], 10, 720.0),  # portatili
    ],
    fecha_esperada="2025-04-01",
    notas="Pedido urgente por stock bajo"
)
print(f"Ordine #{id_pedido} creato")

gprov.recibir_pedido(id_pedido)
print(f"Ordine #{id_pedido} ricevuto e stock aggiornato")

print("\nOrdini:")
display(gprov.listar_pedidos())

print("\nSuggerimenti di riassortimento:")
display(gprov.sugerir_pedidos())

---
## Blocco 5 — Analisi e report

In [ ]:
class Analitica:

    def __init__(self, bd: BaseDeDatos):
        self.bd = bd

    def resumen_general(self) -> dict:
        cur = self.bd.conn.execute("""
            SELECT
                COUNT(*) AS total_productos,
                SUM(CASE WHEN stock_actual <= stock_minimo THEN 1 ELSE 0 END) AS en_alerta,
                ROUND(SUM(precio_venta * stock_actual), 2) AS valor_inventario_venta,
                ROUND(SUM(precio_coste * stock_actual), 2) AS valor_inventario_coste
            FROM productos WHERE activo = 1
        """)
        fila = dict(cur.fetchone())

        cur2 = self.bd.conn.execute(
            "SELECT COUNT(*) AS movimientos_hoy FROM movimientos WHERE fecha = date('now')"
        )
        fila.update(dict(cur2.fetchone()))
        return fila

    def valor_por_categoria(self) -> pd.DataFrame:
        return self.bd.q("""
            SELECT c.nombre AS categoria,
                   COUNT(p.id)                                   AS productos,
                   SUM(p.stock_actual)                           AS unidades,
                   ROUND(SUM(p.precio_coste  * p.stock_actual), 2) AS valor_coste,
                   ROUND(SUM(p.precio_venta  * p.stock_actual), 2) AS valor_venta,
                   ROUND(AVG((p.precio_venta - p.precio_coste) / p.precio_venta * 100), 1) AS margen_medio_pct
            FROM productos p
            JOIN categorias c ON p.categoria_id = c.id
            WHERE p.activo = 1
            GROUP BY c.id, c.nombre
            ORDER BY valor_venta DESC
        """)

    def movimientos_resumen(self) -> pd.DataFrame:
        return self.bd.q("""
            SELECT m.tipo,
                   COUNT(*)        AS operaciones,
                   SUM(m.cantidad) AS unidades_total
            FROM movimientos m
            GROUP BY m.tipo
            ORDER BY operaciones DESC
        """)

    def top_movidos(self, n: int = 5) -> pd.DataFrame:
        return self.bd.q(f"""
            SELECT p.sku, p.nombre,
                   SUM(CASE WHEN m.tipo='salida' THEN m.cantidad ELSE 0 END) AS unidades_salida,
                   SUM(CASE WHEN m.tipo='entrada' THEN m.cantidad ELSE 0 END) AS unidades_entrada,
                   COUNT(DISTINCT m.id) AS num_movimientos
            FROM productos p
            JOIN movimientos m ON p.id = m.producto_id
            GROUP BY p.id, p.sku, p.nombre
            ORDER BY num_movimientos DESC
            LIMIT {n}
        """)

    def exportar_csv(self, ruta: Path):
        df = self.bd.q("""
            SELECT p.sku, p.nombre, c.nombre AS categoria,
                   p.precio_venta, p.precio_coste, p.stock_actual, p.stock_minimo,
                   ROUND((p.precio_venta-p.precio_coste)/p.precio_venta*100,1) AS margen_pct
            FROM productos p JOIN categorias c ON p.categoria_id = c.id
            WHERE p.activo = 1 ORDER BY c.nombre, p.nombre
        """)
        df.to_csv(ruta, index=False)
        return ruta


an = Analitica(bd)

print("=" * 55)
print("    PANNELLO DI CONTROLLO — INVENTARIO")
print("=" * 55)

resumen = an.resumen_general()
print(f"  Prodotti attivi:          {resumen['total_productos']}")
print(f"  In allerta di stock:      {resumen['en_alerta']}")
print(f"  Valore inventario (PVP):  {resumen['valor_inventario_venta']:>12,.2f}€")
print(f"  Valore inventario (costo):{resumen['valor_inventario_coste']:>12,.2f}€")

print("\n--- Valore per categoria ---")
display(an.valor_por_categoria())

print("\n--- Riepilogo movimenti ---")
display(an.movimientos_resumen())

print("\n--- Top 5 prodotti con più movimento ---")
display(an.top_movidos())

csv_path = Path.home() / "inventario_export.csv"
an.exportar_csv(csv_path)
print(f"\nCSV esportato → {csv_path}")

---
## Blocco 6 — Demo del flusso completo

In [ ]:
print("\n" + "=" * 60)
print("    SIMULAZIONE DI UNA GIORNATA DI LAVORO")
print("=" * 60)

# Mattina: vendite
print("\n[08:00] Apertura — Vendite del mattino")
ventas_manana = [
    (ids_prod[0], 2, "Venta tienda"),   # Smartphone
    (ids_prod[1], 5, "Venta online"),   # Auricolari
    (ids_prod[4], 3, "Venta tienda"),   # Tastiere
    (ids_prod[9], 20, "Venta B2B"),     # Penne
]
for pid, cant, mot in ventas_manana:
    gm.salida(pid, cant, mot, "empleado1")
    prod = gp.obtener_producto(pid)
    print(f"  📦 {prod['nombre']:25s} venduti={cant}  stock_residuo={prod['stock_actual']}")

# Mezzogiorno: verifica allerte
print("\n[12:00] Verifica delle allerte")
alertas = gm.alertas_stock()
if alertas.empty:
    print("  ✅ Nessuna allerta")
else:
    print(f"  ⚠️  {len(alertas)} prodotto/i sotto il minimo:")
    for _, fila in alertas.iterrows():
        print(f"      {fila['sku']:8s} {fila['nombre']:25s} stock={fila['stock_actual']} min={fila['stock_minimo']}")

# Pomeriggio: lanciare l'ordine automatico
print("\n[15:00] Generazione ordine automatico di riassortimento")
sugerencias = gprov.sugerir_pedidos()
if not sugerencias.empty:
    # Raggruppare per fornitore
    grupos = sugerencias.groupby('proveedor')
    for prov_nombre, grupo in grupos:
        cur = bd.conn.execute("SELECT id FROM proveedores WHERE nombre = ?", (prov_nombre,))
        prov_id = cur.fetchone()['id']

        lineas_pedido = []
        for _, row in grupo.iterrows():
            cur2 = bd.conn.execute("SELECT id, precio_coste FROM productos WHERE sku = ?", (row['sku'],))
            p = cur2.fetchone()
            lineas_pedido.append((p['id'], int(row['cantidad_sugerida']), p['precio_coste']))

        if lineas_pedido:
            pid = gprov.crear_pedido(prov_id, lineas_pedido, notas="Auto-reposición")
            coste = sum(l[1] * l[2] for l in lineas_pedido)
            print(f"  📋 Ordine #{pid} a '{prov_nombre}': {len(lineas_pedido)} righe, costo stimato={coste:.2f}€")

# Chiusura
print("\n[18:00] Chiusura — Riepilogo della giornata")
resumen = an.resumen_general()
print(f"  Valore inventario attuale (PVP): {resumen['valor_inventario_venta']:>10,.2f}€")
print(f"  Prodotti in allerta:              {resumen['en_alerta']}")
print()
print("Ordini in attesa di ricezione:")
display(gprov.listar_pedidos())

---
## Pratica — Amplia il sistema

---

### Esercizio 1 — Rotazione dell'inventario

**Consegna:** Aggiungi alla classe `Analitica` il metodo `rotacion_inventario()` che calcoli per ogni prodotto:
- Totale unità vendute (uscite).
- Stock medio ((stock_iniziale + stock_finale) / 2).
- **Indice di rotazione** = unità_vendute / stock_medio.
- Classificazione: `lento` (< 1), `normale` (1–3), `veloce` (> 3).

Ordina per indice decrescente.

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 1</summary>

```python
def rotacion_inventario(self) -> pd.DataFrame:
    df = self.bd.q("""
        SELECT p.sku, p.nombre,
               SUM(CASE WHEN m.tipo='salida' THEN m.cantidad ELSE 0 END) AS vendidas,
               SUM(CASE WHEN m.tipo='entrada' THEN m.cantidad ELSE 0 END) AS entradas,
               p.stock_actual AS stock_final
        FROM productos p
        LEFT JOIN movimientos m ON p.id = m.producto_id
        WHERE p.activo = 1
        GROUP BY p.id, p.sku, p.nombre, p.stock_actual
    """)
    df['stock_inicial'] = df['stock_final'] + df['vendidas'] - df['entradas']
    df['stock_medio']   = (df['stock_inicial'] + df['stock_final']) / 2
    df['rotacion']      = (df['vendidas'] / df['stock_medio'].replace(0, float('nan'))).round(2)
    df['clasificacion'] = df['rotacion'].apply(
        lambda r: 'rápido' if r > 3 else ('normal' if r >= 1 else 'lento')
        if pd.notna(r) else 'sin_datos'
    )
    return df[['sku','nombre','vendidas','stock_medio','rotacion','clasificacion']].sort_values('rotacion', ascending=False)

# Applichiamo la patch all'istanza esistente
import types
an.rotacion_inventario = types.MethodType(rotacion_inventario, an)
display(an.rotacion_inventario())
```
</details>

---
### Esercizio 2 — Validare l'integrità referenziale

**Consegna:** Scrivi la funzione `verificar_bd(bd)` che verifichi:
1. Non ci sono prodotti senza categoria.
2. Non ci sono prodotti con prezzo di costo >= prezzo di vendita.
3. Non ci sono movimenti che hanno lasciato lo stock negativo.
4. Non ci sono ordini senza righe d'ordine.

Restituisce un dict con i risultati di ogni verifica.

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 2</summary>

```python
def verificar_bd(bd):
    checks = {}

    # 1. Prodotti senza categoria
    n = bd.conn.execute("SELECT COUNT(*) FROM productos WHERE categoria_id IS NULL").fetchone()[0]
    checks['sin_categoria'] = {'ok': n == 0, 'valor': n}

    # 2. Prezzo costo >= prezzo vendita
    n = bd.conn.execute("SELECT COUNT(*) FROM productos WHERE precio_coste >= precio_venta").fetchone()[0]
    checks['margen_negativo'] = {'ok': n == 0, 'valor': n}

    # 3. Stock negativo
    n = bd.conn.execute("SELECT COUNT(*) FROM productos WHERE stock_actual < 0").fetchone()[0]
    checks['stock_negativo'] = {'ok': n == 0, 'valor': n}

    # 4. Ordini senza righe
    n = bd.conn.execute("""
        SELECT COUNT(*) FROM pedidos_proveedor pp
        LEFT JOIN lineas_pedido lp ON pp.id = lp.pedido_id
        WHERE lp.pedido_id IS NULL
    """).fetchone()[0]
    checks['pedidos_vacios'] = {'ok': n == 0, 'valor': n}

    print("=== Verifica di integrità ===")
    for nombre, resultado in checks.items():
        icono = '✅' if resultado['ok'] else '❌'
        print(f"  {icono} {nombre:22s}: {resultado['valor']} problemi")

    return checks

verificar_bd(bd)
```
</details>

---
### Esercizio 3 — Query analitica complessa

**Consegna:** Scrivi una query SQL che mostri, per fornitore:
- Numero di prodotti che fornisce.
- Valore totale dell'inventario a prezzo di costo.
- Numero di ordini effettuati e il loro importo totale.
- Data dell'ultimo ordine.
- Se ci sono prodotti di quel fornitore in allerta di stock.

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 3</summary>

```python
display(bd.q("""
    SELECT
        pr.nombre                                          AS proveedor,
        COUNT(DISTINCT p.id)                               AS num_productos,
        ROUND(SUM(p.precio_coste * p.stock_actual), 2)     AS valor_stock_coste,
        COUNT(DISTINCT pp.id)                              AS num_pedidos,
        ROUND(SUM(lp.cantidad * lp.precio_ud), 2)          AS importe_pedidos,
        MAX(pp.fecha_pedido)                               AS ultimo_pedido,
        SUM(CASE WHEN p.stock_actual <= p.stock_minimo THEN 1 ELSE 0 END) AS prods_en_alerta
    FROM proveedores pr
    LEFT JOIN productos         p  ON p.proveedor_id  = pr.id
    LEFT JOIN pedidos_proveedor pp ON pp.proveedor_id = pr.id
    LEFT JOIN lineas_pedido     lp ON lp.pedido_id    = pp.id
    GROUP BY pr.id, pr.nombre
    ORDER BY valor_stock_coste DESC
"""))
```
</details>

---
### Esercizio 4 — Trasferimento tra magazzini

**Consegna:** Il sistema attuale ha un unico magazzino. Aggiungi il supporto per più ubicazioni:

1. Progetta e crea la tabella `ubicaciones` (id, nombre, descripcion).
2. Aggiungi una colonna `ubicacion_id` alla tabella `movimientos`.
3. Implementa la funzione `transferir(producto_id, cantidad, de_ubicacion, a_ubicacion)` che registri l'uscita da un magazzino e l'entrata in un altro come transazione atomica.

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 4</summary>

```python
# 1 e 2. Creare la tabella e aggiungere la colonna
bd.ejecutar("""
    CREATE TABLE IF NOT EXISTS ubicaciones (
        id          INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre      TEXT NOT NULL UNIQUE,
        descripcion TEXT
    )
""")
try:
    bd.ejecutar("ALTER TABLE movimientos ADD COLUMN ubicacion_id INTEGER REFERENCES ubicaciones(id)")
except Exception:
    pass  # esiste già

# Creare ubicazioni di esempio
bd.ejecutar("INSERT OR IGNORE INTO ubicaciones (nombre) VALUES ('Almacén Central')")
bd.ejecutar("INSERT OR IGNORE INTO ubicaciones (nombre) VALUES ('Tienda Madrid')")
bd.ejecutar("INSERT OR IGNORE INTO ubicaciones (nombre) VALUES ('Tienda Barcelona')")

# 3. Funzione di trasferimento
def transferir(bd, producto_id, cantidad, de_ubicacion_nombre, a_ubicacion_nombre):
    def get_ub_id(nombre):
        r = bd.conn.execute("SELECT id FROM ubicaciones WHERE nombre = ?", (nombre,)).fetchone()
        if not r: raise ValueError(f"Ubicazione '{nombre}' non trovata")
        return r['id']

    id_origen  = get_ub_id(de_ubicacion_nombre)
    id_destino = get_ub_id(a_ubicacion_nombre)

    with bd.conn:
        bd.conn.execute("""
            INSERT INTO movimientos (producto_id, tipo, cantidad, motivo, ubicacion_id)
            VALUES (?, 'salida', ?, 'Transferencia a ' || ?, ?)
        """, (producto_id, cantidad, a_ubicacion_nombre, id_origen))
        bd.conn.execute("""
            INSERT INTO movimientos (producto_id, tipo, cantidad, motivo, ubicacion_id)
            VALUES (?, 'entrada', ?, 'Transferencia desde ' || ?, ?)
        """, (producto_id, cantidad, de_ubicacion_nombre, id_destino))

    print(f"Trasferite {cantidad} unità da '{de_ubicacion_nombre}' → '{a_ubicacion_nombre}'")

transferir(bd, ids_prod[0], 5, "Almacén Central", "Tienda Madrid")
transferir(bd, ids_prod[0], 3, "Almacén Central", "Tienda Barcelona")

# Verifichiamo lo storico con le ubicazioni
display(bd.q("""
    SELECT m.fecha, m.tipo, m.cantidad, m.motivo, u.nombre AS ubicacion
    FROM movimientos m
    LEFT JOIN ubicaciones u ON m.ubicacion_id = u.id
    WHERE m.producto_id = ?
    ORDER BY m.id DESC
""", (ids_prod[0],)))
```
</details>

---
### Esercizio 5 (sfida) — CLI del sistema

**Consegna:** Implementa un'interfaccia a riga di comando (simulata nel notebook come funzione) `cli_inventario(comando, *args)` che supporti:

```
cli_inventario('listar')                    → tabella dei prodotti
cli_inventario('buscar', 'Teclado')         → prodotti il cui nome contiene la ricerca
cli_inventario('vender', 'IN-002', 5)       → registra la vendita di 5 unità dello SKU
cli_inventario('reponer', 'IN-002', 50)     → registra un'entrata di stock
cli_inventario('alertas')                   → prodotti in allerta
cli_inventario('informe')                   → dashboard completo
cli_inventario('exportar', 'archivo.csv')   → esporta l'inventario in CSV
```

La CLI deve gestire gli errori con messaggi chiari (prodotto non trovato, stock insufficiente, ecc.).

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 5</summary>

```python
def cli_inventario(comando, *args):
    """Interfaccia a riga di comando del sistema di inventario."""
    def get_id_por_sku(sku):
        cur = bd.conn.execute("SELECT id FROM productos WHERE sku = ?", (sku,))
        r = cur.fetchone()
        if not r:
            raise ValueError(f"SKU '{sku}' non trovato")
        return r['id']

    try:
        if comando == 'listar':
            display(gp.listar_productos())

        elif comando == 'buscar':
            termino = args[0] if args else ''
            display(bd.q("""
                SELECT p.sku, p.nombre, c.nombre AS cat, p.precio_venta, p.stock_actual
                FROM productos p JOIN categorias c ON p.categoria_id = c.id
                WHERE p.nombre LIKE ? AND p.activo = 1
                ORDER BY p.nombre
            """, (f'%{termino}%',)))

        elif comando == 'vender':
            sku, cantidad = args[0], int(args[1])
            pid = get_id_por_sku(sku)
            gm.salida(pid, cantidad, "Venta CLI", "cli")
            prod = gp.obtener_producto(pid)
            print(f"✅ Vendute {cantidad} unità di '{prod['nombre']}' — stock={prod['stock_actual']}")

        elif comando == 'reponer':
            sku, cantidad = args[0], int(args[1])
            pid = get_id_por_sku(sku)
            gm.entrada(pid, cantidad, "Reposición CLI", "cli")
            prod = gp.obtener_producto(pid)
            print(f"✅ Aggiunte {cantidad} unità a '{prod['nombre']}' — stock={prod['stock_actual']}")

        elif comando == 'alertas':
            alertas = gm.alertas_stock()
            if alertas.empty:
                print("✅ Nessuna allerta di stock")
            else:
                print(f"⚠️ {len(alertas)} prodotto/i in allerta:")
                display(alertas)

        elif comando == 'informe':
            r = an.resumen_general()
            print(f"Prodotti: {r['total_productos']} | Allerte: {r['en_alerta']} | Valore PVP: {r['valor_inventario_venta']:,.2f}€")
            display(an.valor_por_categoria())

        elif comando == 'exportar':
            ruta = Path.home() / (args[0] if args else 'inventario.csv')
            an.exportar_csv(ruta)
            print(f"✅ Esportato in {ruta}")

        else:
            print(f"❌ Comando sconosciuto: '{comando}'")
            print("Comandi validi: listar, buscar, vender, reponer, alertas, informe, exportar")

    except ValueError as e:
        print(f"❌ Errore: {e}")
    except Exception as e:
        print(f"❌ Errore imprevisto: {e}")


# Demo
cli_inventario('buscar', 'teclado')
cli_inventario('vender', 'IN-002', 3)
cli_inventario('alertas')
cli_inventario('informe')
cli_inventario('vender', 'IN-002', 999)  # errore: stock insufficiente
cli_inventario('listar_todo')            # errore: comando sconosciuto
```
</details>

---
## Cosa abbiamo costruito

```
SISTEMA DI INVENTARIO — COMPONENTI
────────────────────────────────────────────────────────────
BaseDeDatos         DDL, connessione, esecuzione query
GestorProductos     CRUD prodotti e categorie, margine
GestorMovimientos   Entrate, uscite, rettifiche, allerte
GestorProveedores   Ordini, ricezione, suggerimenti automatici
Analitica           Report, valore per categoria, esportazione CSV
────────────────────────────────────────────────────────────

CONCETTI DEL MODULO APPLICATI
────────────────────────────────────────────────────────────
DDL         CREATE TABLE, FOREIGN KEY, CHECK, AUTOINCREMENT
DML         INSERT, UPDATE (dentro transazioni)
SELECT      WHERE, JOIN (3 tabelle), GROUP BY, HAVING, subquery
Progettazione  1FN/2FN/3FN, relazioni 1:N e N:M, tabella ponte
Python      sqlite3, row_factory, context manager, pandas
────────────────────────────────────────────────────────────
```

## Cosa vedremo nel Modulo 5?

Il **Modulo 5** ci introduce allo sviluppo web con Python: Flask, API REST, autenticazione e deployment.

In [ ]:
bd.cerrar()
print("Sistema di inventario chiuso")
print(f"Database salvato in: {DB_PATH}")